# Đánh giá kiểm tra ngắn: HPA và autoscaler trên cùng trace tải

**Mục tiêu:** biến dữ liệu thô của kiểm tra ngắn (mục 10 của `DEPLOY_AUTOSCALER_MODELS.md`) thành **bảng chỉ số và hình chuỗi thời gian** so sánh HPA với autoscaler GAT-GRU + PPO trên **cùng một trace tải**: autoscaler có dùng ít replica hơn, scale sớm hơn và giữ độ trễ thấp hơn HPA hay không.

Notebook **không tham gia triển khai**: chạy trên máy build, sau khi `autoscaler/scripts/run-comparison.sh` chạy xong và kết quả đã được `rsync` về `dataset/raw-eval/`. Notebook chỉ đọc dữ liệu, không sửa.

```text
run-comparison.sh ─► (loadgen) ~/dataset/raw-eval/<kịch_bản>-seed<seed>-900s/<ngữ_cảnh>/<kịch_bản>/run_XX/
                         │  rsync về máy build: dataset/raw-eval/
                         ▼
             notebook này ─► bảng chỉ số, chênh lệch so với hpa
                           ─► <root>/report/<kịch_bản>_summary.csv
                           ─► <root>/report/<kịch_bản>_timeline.png
```

**Đầu vào của mỗi run:**

| File | Lấy gì |
|---|---|
| `meta.json` | Thời điểm bắt đầu, độ dài run, seed, nhãn ngữ cảnh, số replica lúc bắt đầu |
| `designed_rate.csv` | Đường tải thiết kế (một dòng mỗi giây), dùng để xác định đợt tăng tải |
| `node_metrics.csv` | Tổng replica sẵn sàng và replica mong muốn của 10 service (bỏ `redis-cart`) |
| `service_metrics.csv` | p95 frontend theo từng bước 10 s (từ Jaeger) |
| `locust_stats.csv` | p95 phía client và tỷ lệ lỗi của cả run |
| `driver_report.json` | Số arrival bộ phát tải bỏ (`dropped`) |

Chỉ run có `collect_report.json` mới được coi là hoàn chỉnh.

In [ ]:
import csv
import json
import math
import os
import statistics
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'autoscaler').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'autoscaler').exists():
    raise FileNotFoundError('Cannot locate repository root')

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

## 1. Cấu hình

- `ROOTS`: các thư mục gốc do `run-comparison.sh` tạo, mỗi thư mục ứng với một (kịch bản, seed, độ dài). Mặc định lấy mọi thư mục trong `dataset/raw-eval/`.
- `SURGE_FRAC`: đợt tăng tải là các giây có tải thiết kế ≥ `SURGE_FRAC` × đỉnh.
- `SLO_MS`: ngưỡng p95 frontend để đếm bước vi phạm.
- `OUT_DIR`: `None` thì ghi vào `<root>/report/`.

In [ ]:
EVAL_DIR = ROOT / 'dataset' / 'raw-eval'
ROOTS = sorted(p for p in EVAL_DIR.glob('*-seed*-*s') if p.is_dir())
# Ví dụ chọn tay:
# ROOTS = [EVAL_DIR / 'mmpp-seed2030-900s', EVAL_DIR / 'nhpp-seed2016-900s']
SURGE_FRAC = 0.6
SLO_MS = 300.0
OUT_DIR = None

FIXED = {'redis-cart'}      # không được scale: bỏ khỏi tổng replica
SURGE_WINDOW = 120          # giây đầu đợt tăng tải dùng để đo p95

if not ROOTS:
    raise FileNotFoundError(f'Không thấy thư mục kết quả trong {EVAL_DIR}: rsync từ loadgen trước')
print('ROOTS:', [p.name for p in ROOTS])

## 2. Đọc dữ liệu run

Độ trễ theo thời gian lấy từ `service_metrics.csv` (p95 span server của frontend trong Jaeger, cửa sổ 10 s). **Không** dùng percentile trong `locust_stats_history.csv`: đó là giá trị lũy kế từ đầu run (giá trị cuối trùng p95 cả run trong `locust_stats.csv`), không phản ánh từng thời điểm.

In [ ]:
def read_csv(path):
    with open(path, newline='', encoding='utf-8') as fh:
        return list(csv.DictReader(fh))


def fnum(x):
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None
    return None if math.isnan(v) or math.isinf(v) else v


def load_run(run_dir):
    meta = json.load(open(os.path.join(run_dir, 'meta.json'), encoding='utf-8'))
    t0 = int(meta['start'])
    dur = int(meta.get('duration_sec') or (int(meta['end']) - t0))
    step = int(meta.get('step_sec', 10))
    rate = [fnum(r.get('expected_rps')) or 0.0 for r in read_csv(os.path.join(run_dir, 'designed_rate.csv'))]

    rep, des = defaultdict(float), defaultdict(float)
    svc_des = defaultdict(dict)
    for r in read_csv(os.path.join(run_dir, 'node_metrics.csv')):
        if r.get('service') in FIXED:
            continue
        t = int(float(r['timestamp'])) - t0
        if t < 0 or t >= dur:
            continue
        v, d = fnum(r.get('replicas')), fnum(r.get('replicas_desired'))
        if v is not None:
            rep[t] += v
        if d is not None:
            des[t] += d
            svc_des[r['service']][t] = d

    p95 = {}
    for r in read_csv(os.path.join(run_dir, 'service_metrics.csv')):
        if r.get('service') != 'frontend':
            continue
        t = int(float(r['timestamp'])) - t0
        v = fnum(r.get('latency_p95_ms'))
        if v is not None and 0 <= t < dur:
            p95[t] = v

    agg = next((r for r in read_csv(os.path.join(run_dir, 'locust_stats.csv')) if r.get('Name') == 'Aggregated'), {})
    drv_path = os.path.join(run_dir, 'driver_report.json')
    drv = json.load(open(drv_path, encoding='utf-8')) if os.path.exists(drv_path) else {}
    return {'dir': run_dir, 'meta': meta, 't0': t0, 'dur': dur, 'step': step, 'rate': rate,
            'rep': dict(rep), 'des': dict(des), 'svc_des': svc_des, 'p95': p95,
            'agg': agg, 'drv': drv}


def discover(root):
    # -> {kịch_bản: {ngữ_cảnh: [run_dir, ...]}}; chỉ lấy run có collect_report.json
    found = defaultdict(lambda: defaultdict(list))
    for ctx in sorted(os.listdir(root)):
        cdir = os.path.join(root, ctx)
        if not os.path.isdir(cdir):
            continue
        for scen in sorted(os.listdir(cdir)):
            sdir = os.path.join(cdir, scen)
            if not os.path.isdir(sdir):
                continue
            for run in sorted(os.listdir(sdir)):
                rdir = os.path.join(sdir, run)
                if run.startswith('run_') and os.path.exists(os.path.join(rdir, 'collect_report.json')):
                    found[scen][ctx].append(rdir)
    return found

## 3. Chỉ số

Đợt tăng tải: các giây có tải thiết kế ≥ `SURGE_FRAC` × đỉnh; giây đầu là `onset`, giây cuối là `peak_end`.

| Nhóm | Chỉ số | Cách tính | Tốt khi |
|---|---|---|---|
| Chi phí | `replica_min` | Tổng replica × thời gian (replica-phút) của 10 service | Thấp |
| | `replica_min_post_peak` | Replica-phút sau `peak_end` (HPA giữ replica 5 phút sau khi tải giảm) | Thấp |
| | `scale_changes` | Số lần đổi replica mong muốn, cộng mọi service | Thấp (ổn định) |
| Tính chủ động | `first_scale_up_s` | Lần tăng replica mong muốn đầu tiên so với `onset` (âm = scale trước tải) | Nhỏ / âm |
| | `time_to_capacity_s` | Từ `onset` đến khi tổng replica sẵn sàng đạt R* | Ngắn |
| Chất lượng dịch vụ | `p95_surge_mean_ms`, `p95_surge_max_ms` | p95 frontend trong 120 s đầu đợt tăng tải | Thấp |
| | `p95_converge_s` | Thời gian p95 về lại ≤ 1,2 × mức trước đợt và giữ 30 s | Ngắn |
| | `slo_viol_pct` | % bước 10 s có p95 frontend > `SLO_MS` | Thấp |
| | `p95_run_ms`, `fail_pct` | p95 phía client và tỷ lệ lỗi của cả run (`locust_stats.csv`) | Thấp |

**R\***: mức nhỏ nhất trong các tổng replica cao nhất mà mỗi ngữ cảnh đạt được trong đợt tăng tải, nên mọi ngữ cảnh đều đạt và `time_to_capacity_s` so sánh được.

In [ ]:
def surge_bounds(rate, frac):
    peak = max(rate) if rate else 0.0
    idx = [i for i, v in enumerate(rate) if v >= frac * peak]
    return (idx[0], idx[-1], peak) if idx else (None, None, peak)


def mean_or_none(xs):
    return statistics.fmean(xs) if xs else None


def first_scale_up(des, onset, look_back=120):
    ts = sorted(des)
    if not ts:
        return None
    before = [t for t in ts if t <= onset - look_back]
    base = des[before[-1]] if before else des[ts[0]]
    for t in ts:
        if t >= onset - look_back and des[t] > base:
            return t - onset
    return None


def p95_convergence(p95, onset, factor=1.2, hold=30, degrade_within=180):
    pre = [v for t, v in p95.items() if onset - 90 <= t < onset]
    if not pre:
        return None
    base = statistics.median(pre)
    ts = sorted(t for t in p95 if t >= onset)
    degraded = next((t for t in ts if t <= onset + degrade_within and p95[t] > factor * base), None)
    if degraded is None:
        return 0                                   # p95 không xấu đi đáng kể
    start = None
    for t in ts:
        if t < degraded:
            continue
        if p95[t] <= factor * base:
            start = t if start is None else start
            if t - start >= hold:
                return start - onset
        else:
            start = None
    return None                                    # không hội tụ trong run


def time_to_capacity(rep, onset, target):
    for t in sorted(rep):
        if t >= onset and rep[t] >= target:
            return t - onset
    return None


def analyse(run, frac, slo_ms):
    onset, peak_end, peak = surge_bounds(run['rate'], frac)
    step = run['step']
    rep, des = run['rep'], run['des']
    out = {
        'seed': run['meta'].get('seed'),
        'label': run['meta'].get('autoscaler'),
        'onset_s': onset, 'peak_end_s': peak_end, 'peak_rps': round(peak, 1),
        'replica_min': round(sum(rep.values()) * step / 60, 1),
        'replica_min_post_peak': round(sum(v for t, v in rep.items() if peak_end is not None and t > peak_end) * step / 60, 1),
        'scale_changes': sum(
            sum(1 for a, b in zip(sorted(s), sorted(s)[1:]) if s[a] != s[b]) for s in run['svc_des'].values()),
        'max_total_replicas': max(rep.values()) if rep else None,
        'p95_run_ms': fnum(run['agg'].get('95%')),
        'fail_pct': None,
        'dropped': run['drv'].get('dropped'),
        'replicas_at_start': run['meta'].get('replicas_at_start'),
    }
    req, fail = fnum(run['agg'].get('Request Count')), fnum(run['agg'].get('Failure Count'))
    if req:
        out['fail_pct'] = round(100.0 * (fail or 0) / req, 3)
    if run['p95']:
        out['slo_viol_pct'] = round(100.0 * sum(v > slo_ms for v in run['p95'].values()) / len(run['p95']), 1)
    if onset is not None:
        win95 = [v for t, v in run['p95'].items() if onset <= t < onset + SURGE_WINDOW]
        out['p95_surge_mean_ms'] = round(mean_or_none(win95), 0) if win95 else None
        out['p95_surge_max_ms'] = max(win95) if win95 else None
        out['first_scale_up_s'] = first_scale_up(des, onset)
        out['p95_converge_s'] = p95_convergence(run['p95'], onset)
        out['max_rep_in_surge'] = max((v for t, v in rep.items() if onset <= t <= (peak_end or onset) + 60), default=None)
    return out

## 4. Đánh giá theo kịch bản và kiểm tra tính hợp lệ

Cặp so sánh chỉ hợp lệ khi các ngữ cảnh chạy **cùng trace** và **cùng trạng thái ban đầu**. Notebook in `CẢNH BÁO` khi:

| Điều kiện | Ý nghĩa |
|---|---|
| Seed khác nhau giữa các run | Trace tải khác nhau, không so sánh được |
| `dropped > 0` | Bộ phát tải bỏ request, tải thực tế không còn giống nhau |
| `replicas_at_start ≠ 1` | Các ngữ cảnh không bắt đầu từ cùng trạng thái |
| Nhãn chứa `dummy` | Dùng PPO mẫu: chỉ chứng minh đường ống, không quy cải thiện cho PPO |

In [ ]:
COLS = [('label', 'nhãn'), ('seed', 'seed'), ('replica_min', 'replica-phút'),
        ('replica_min_post_peak', 'replica-phút sau đỉnh'), ('scale_changes', 'lần scale'),
        ('first_scale_up_s', 'scale-up đầu (s)'), ('time_to_capacity_s', 'đạt R* (s)'),
        ('p95_surge_mean_ms', 'p95 FE đầu đợt TB'), ('p95_surge_max_ms', 'p95 FE đầu đợt max'),
        ('p95_converge_s', 'p95 hội tụ (s)'), ('slo_viol_pct', 'bước p95 FE > SLO %'),
        ('p95_run_ms', 'p95 client cả run'), ('fail_pct', 'lỗi %')]
LOWER_IS_BETTER = ['replica_min', 'replica_min_post_peak', 'scale_changes', 'time_to_capacity_s',
                   'p95_surge_mean_ms', 'p95_surge_max_ms', 'p95_converge_s', 'slo_viol_pct', 'p95_run_ms']


def evaluate_scenario(by_ctx):
    order = sorted(by_ctx, key=lambda c: (c != 'hpa', c))          # hpa đứng đầu làm mốc
    runs_by_ctx = {c: [load_run(d) for d in by_ctx[c]] for c in order}
    res = {c: [analyse(r, SURGE_FRAC, SLO_MS) for r in runs_by_ctx[c]] for c in order}

    peaks = [x.get('max_rep_in_surge') for c in order for x in res[c] if x.get('max_rep_in_surge')]
    target = min(peaks) if peaks else None                          # R*
    for c in order:
        for run, x in zip(runs_by_ctx[c], res[c]):
            x['time_to_capacity_s'] = (time_to_capacity(run['rep'], x['onset_s'], target)
                                       if target and x.get('onset_s') is not None else None)

    warn = []
    seeds = {x['seed'] for c in order for x in res[c]}
    if len(seeds) > 1:
        warn.append(f'seed khác nhau giữa các run {sorted(seeds)}: KHÔNG cùng trace')
    for c in order:
        for x in res[c]:
            if x.get('dropped'):
                warn.append(f"{c}: driver bỏ {x['dropped']} arrival (tải không còn giống nhau)")
            if x.get('replicas_at_start') not in (1, None):
                warn.append(f"{c}: replicas_at_start = {x['replicas_at_start']} (không bắt đầu từ 1)")
            if x.get('label') and 'dummy' in str(x['label']):
                warn.append(f'{c}: dùng PPO mẫu, chỉ chứng minh đường ống')
    return order, runs_by_ctx, res, target, warn


def summary_table(order, res):
    rows = [{'ngữ cảnh': c, **{h: x.get(k) for k, h in COLS}} for c in order for x in res[c]]
    return pd.DataFrame(rows).set_index('ngữ cảnh')


def delta_vs_hpa(order, res):
    # Chênh lệch % so với hpa (trung bình theo ngữ cảnh); âm = tốt hơn hpa
    if 'hpa' not in order or len(order) < 2:
        return None
    base = {k: mean_or_none([x[k] for x in res['hpa'] if x.get(k) is not None]) for k in LOWER_IS_BETTER}
    out = {}
    for c in order[1:]:
        row = {}
        for k in LOWER_IS_BETTER:
            v = mean_or_none([x[k] for x in res[c] if x.get(k) is not None])
            b = base.get(k)
            row[k] = None if v is None or b in (None, 0) else round(100 * (v - b) / b, 1)
        out[c] = row
    return pd.DataFrame(out).T


def save_csv(path, order, res):
    rows = [{'context': c, **{k: x.get(k) for k, _ in COLS}} for c in order for x in res[c]]
    with open(path, 'w', newline='', encoding='utf-8') as fh:
        w = csv.DictWriter(fh, fieldnames=['context'] + [k for k, _ in COLS])
        w.writeheader()
        w.writerows(rows)

## 5. Hình chuỗi thời gian

Ba tầng dùng chung trục thời gian: tải thiết kế, tổng replica sẵn sàng của 10 service, p95 frontend (Jaeger, cửa sổ 10 s). Vùng cam là 120 s đầu đợt tăng tải, nét đỏ là lúc tải hết đỉnh. Mỗi ngữ cảnh vẽ run đầu tiên.

In [ ]:
def plot_timeline(scen, runs_by_ctx, r0, path):
    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    first = next(iter(runs_by_ctx.values()))[0]
    axes[0].plot(range(len(first['rate'])), first['rate'], color='gray')
    axes[0].set_ylabel('tải thiết kế (rps)')
    for ctx, runs in runs_by_ctx.items():
        run = runs[0]
        ts = sorted(run['rep'])
        axes[1].step(ts, [run['rep'][t] for t in ts], where='post', label=ctx)
        pts = sorted(run['p95'])
        axes[2].plot(pts, [run['p95'][t] for t in pts], label=ctx, linewidth=1)
    for ax in axes:
        if r0.get('onset_s') is not None:
            ax.axvspan(r0['onset_s'], r0['onset_s'] + SURGE_WINDOW, color='orange', alpha=0.15)
        if r0.get('peak_end_s') is not None:
            ax.axvline(r0['peak_end_s'], color='red', linestyle='--', linewidth=0.8)
    axes[1].set_ylabel('tổng replica sẵn sàng\n(10 service)')
    axes[2].set_ylabel('p95 frontend (ms)\nJaeger, cửa sổ 10 s')
    axes[2].set_xlabel('thời gian từ lúc bắt đầu run (s)')
    axes[1].legend()
    axes[2].legend()
    fig.suptitle(f'{scen}: cùng trace, so sánh ngữ cảnh')
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.show()

## 6. Kết quả

In [ ]:
for root in ROOTS:
    found = discover(root)
    if not found:
        print(f'[{root.name}] không thấy run hoàn chỉnh (cần collect_report.json)')
        continue
    out_dir = Path(OUT_DIR) if OUT_DIR else root / 'report'
    out_dir.mkdir(parents=True, exist_ok=True)
    for scen, by_ctx in found.items():
        order, runs_by_ctx, res, target, warn = evaluate_scenario(by_ctx)
        r0 = res[order[0]][0]
        print(f'\n=== {root.name} / {scen} ===')
        print(f"Đợt tăng tải (tải thiết kế >= {SURGE_FRAC:.0%} đỉnh {r0['peak_rps']} rps): "
              f"bắt đầu {r0['onset_s']} s, kết thúc {r0['peak_end_s']} s; R* = {target} replica")
        for w in warn:
            print(f'  CẢNH BÁO: {w}')
        display(summary_table(order, res))
        delta = delta_vs_hpa(order, res)
        if delta is not None:
            print('Chênh lệch so với hpa (%, âm = tốt hơn):')
            display(delta)
        csv_path = out_dir / f'{scen}_summary.csv'
        save_csv(csv_path, order, res)
        print('bảng:', csv_path)
        png_path = out_dir / f'{scen}_timeline.png'
        plot_timeline(scen, runs_by_ctx, r0, png_path)
        print('hình:', png_path)

## 7. Lưu ý khi diễn giải

- **Nghiên cứu tình huống, không phải kiểm định thống kê**: mỗi kịch bản chỉ có một cặp run. Kết luận ở dạng "trên trace X, autoscaler dùng ít hơn N replica-phút và p95 đầu đợt thấp hơn M%", không khái quát hóa.
- **Mẫu Jaeger thưa**: độ phủ Jaeger của dataset khoảng 11–16% (bảng chất lượng trong `preprocess_lstm`). Nếu kiểm tra ngắn có độ phủ tương tự, p95 mỗi bước 10 s tính trên ít span nên nhiễu; ưu tiên giá trị trung bình và xu hướng hơn một giá trị max đơn lẻ.
- **`slo_viol_pct` cần đọc thận trọng**: p95 frontend của dataset thường đã vượt 300 ms (bão hòa), nên chỉ số này có thể ít phân biệt giữa các ngữ cảnh.
- **Hình chỉ vẽ run đầu tiên** của mỗi ngữ cảnh; nếu có nhiều lần lặp, các run sau chỉ có trong bảng.
- **`designed_rate.csv` được giả định một dòng mỗi giây**: chỉ số dòng được dùng làm số giây từ đầu run.
- **PPO mẫu** (nhãn `gatgru-ppo-dummy`) chỉ chứng minh đường ống chạy; không quy cải thiện cho PPO.